<div style="color:#3c4d5a; border-top:7px solid #42A5F5; border-bottom:7px solid #42A5F5; padding:8px; text-align:center; text-transform:uppercase">
  <h1>STREAMML - ENTRENAMIENTO Y CREACION DEL AGENTE</h1>
</div>

<strong>Proyecto:</strong> StreamML  |    <strong>Integrantes:</strong> Alexis Guaman y Cinthya Ramon  |    <strong>Modalidad:</strong> Machine Learning offline

<div id="objetivo" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Objetivo del agente</h2> </div>

Construir de principio a fin dos clasificadores supervisados y conectarlos con el agente autónomo de StreamML. El modelo reactivo recomienda `low`, `medium` o `high`; el predictivo anticipa `maintain` o `downgrade_needed`; el agente transforma ambas salidas en acciones seguras, explicables y con memoria operacional.

**Aclaración técnica:** el agente no es un tercer modelo ni consume una API externa. Los modelos se entrenan; el agente se crea como una política determinista propia con margen de capacidad, histéresis, cooldown, respaldo y recuperación.

<div id="requisitos" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Requisitos, entradas y salida esperada</h2> </div>

Se utiliza Python 3.11, pandas, NumPy, scikit-learn y joblib. Las entradas son los CSV procesados, contratos versionados y splits oficiales por sesión. La ejecución entrena candidatos en memoria, ajusta hiperparámetros sin consultar test, evalúa contra baselines, guarda y recarga copias temporales, y genera decisiones reales del agente sin modificar el registro oficial.

<div id="contenido" style="color:#106ba3"><h3>Contenido</h3> </div>

- [Objetivo del agente](#objetivo)
- [Fundamentos metodologicos](#fundamentos-metodologicos)
- [Fase 1: Entorno reproducible](#fase-1-entorno)
- [Fase 2: Preparacion y validacion](#fase-2-preparacion)
- [Fase 3: Variables y preprocesamiento](#fase-3-variables)
- [Fase 4: Division train, validacion y test](#fase-4-division)
- [Fase 5: Seleccion, entrenamiento e hiperparametros](#fase-5-entrenamiento)
- [Fase 6: Evaluacion y modelos base](#fase-6-evaluacion)
- [Fase 7: Guardado y carga](#fase-7-guardado)
- [Fase 8: Integracion con el agente](#fase-8-agente)
- [Fase 9: Ejemplos reales](#fase-9-ejemplos)
- [Fase 10: Errores y casos limite](#fase-10-errores)
- [Conclusiones y proximos pasos](#conclusiones)

<div id="fundamentos-metodologicos" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Fundamentos metodologicos</h2> </div>

Los splits se agrupan por `session_id` para impedir que ventanas de una misma sesión crucen etapas. El ajuste usa solo train; validación selecciona configuración y threshold; test se consulta una vez al final. Macro F1 y balanced accuracy evitan ocultar la clase minoritaria, mientras DummyClassifier establece el mínimo comparable.

Este notebook entrena candidatos compactos de regresión logística para demostrar todo el ciclo sin sobrescribir la publicación oficial. `02_model_training.ipynb` conserva la comparación completa con árboles y random forest.

<div id="fase-1-entorno" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Fase 1: Entorno reproducible</h2> </div>

In [1]:
from pathlib import Path
import json
import shutil
import sys
import tempfile

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, GroupKFold, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.streamml.agent import AgentInput, AgentPolicy, AgentState, AutonomousStreamingAgent
from src.streamml.services.release import classification_metrics as reactive_metrics
from src.streamml.training.predictive import (
    choose_threshold,
    classification_metrics as predictive_metrics,
    positive_probability,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")
print("Proyecto:", ROOT.name, "| random_state:", RANDOM_STATE)

Proyecto: Adaptive-Streaming-ai | random_state: 42


<div id="fase-2-preparacion" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Fase 2: Preparacion y validacion de datos</h2> </div>

Se cargan los datasets y contratos reales. No se imputan valores silenciosamente: cualquier ausencia, infinito, columna faltante o etiqueta desconocida detiene la ejecución. La auditoría persistida añade controles de duplicación, solapamiento y variables constantes.

In [2]:
reactive = pd.read_csv(ROOT / "data" / "processed" / "reactive_dataset.csv")
predictive = pd.read_csv(ROOT / "data" / "processed" / "predictive_dataset.csv")
reactive_contract = json.loads((ROOT / "src" / "streamml" / "config" / "reactive_feature_contract.json").read_text(encoding="utf-8"))
predictive_contract = json.loads((ROOT / "src" / "streamml" / "config" / "predictive_feature_contract.json").read_text(encoding="utf-8"))
predictive_manifest = json.loads((ROOT / "models" / "registry" / "predictive" / "training_manifest.json").read_text(encoding="utf-8"))
quality_audit = json.loads((ROOT / "reports" / "ml_data_quality.json").read_text(encoding="utf-8"))

reactive_features = reactive_contract["features"]
predictive_features = predictive_contract["features"]
for frame, features, target in [
    (reactive, reactive_features, "target"),
    (predictive, predictive_features, "target_code"),
]:
    assert set(features + [target, "session_id"]).issubset(frame.columns)
    assert not frame[features + [target]].isna().any().any()
    assert np.isfinite(frame[features].to_numpy(dtype=float)).all()

assert set(reactive["target"]) == {"low", "medium", "high"}
assert set(predictive["target_code"]) == {0, 1}
display(pd.DataFrame([
    {"dataset": "reactive", "rows": len(reactive), "sessions": reactive["session_id"].nunique(), "features": len(reactive_features)},
    {"dataset": "predictive", "rows": len(predictive), "sessions": predictive["session_id"].nunique(), "features": len(predictive_features)},
]))
display(pd.DataFrame(quality_audit["predictive"]["warnings"])[["severity", "code", "message"]])

,dataset,rows,sessions,features
0,reactive,26686,26686,3
1,predictive,4336,17,19


,severity,code,message
0,high,duplicate_feature_vectors,Varias ventanas contienen entradas idénticas; ...
1,high,high_window_overlap,La mayoría de las ventanas adyacentes se solap...
2,high,cross_split_feature_overlap,Existen vectores idénticos entre splits aunque...
3,high,class_imbalance,La clase minoritaria representa menos del 15% ...
4,medium,constant_features,Estas variables son constantes y no discrimina...


<div id="fase-3-variables" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Fase 3: Preprocesamiento y generacion de caracteristicas</h2> </div>

Las tres variables reactivas ya son numéricas. El predictivo contiene 19 resúmenes de la ventana histórica: tendencia, dispersión, percentiles, proporciones bajo capacidad y perfil actual. `StandardScaler` se ajusta dentro de un `Pipeline`, exclusivamente con cada fold de entrenamiento, para evitar fuga durante la validación cruzada.

Las variables constantes se mantienen para respetar el contrato publicado, aunque no añaden señal en el dataset actual. Las columnas futuras y el target están prohibidos como entradas.

In [3]:
feature_contracts = pd.DataFrame([
    {"role": "reactive", "feature_count": len(reactive_features), "features": ", ".join(reactive_features)},
    {"role": "predictive", "feature_count": len(predictive_features), "features": ", ".join(predictive_features)},
])
display(feature_contracts)
assert set(predictive_contract["forbidden_input_columns"]).isdisjoint(predictive_features)
assert predictive_contract["lookback_seconds"] == 600
assert predictive_contract["future_horizon_seconds"] == 600

,role,feature_count,features
0,reactive,3,"upload_mbps, download_mbps, latency_ms"
1,predictive,19,"throughput_mean, throughput_median, throughput..."


<div id="fase-4-division" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Fase 4: Division de entrenamiento, validacion y prueba</h2> </div>

In [4]:
reactive_parts = {name: reactive.loc[reactive["split"] == name].copy() for name in ["train", "validation", "test"]}
predictive_parts = {
    name: predictive.loc[predictive["session_id"].isin(predictive_manifest["splits"][name])].copy()
    for name in ["train", "validation", "test"]
}

for parts in [reactive_parts, predictive_parts]:
    session_sets = {name: set(frame["session_id"].astype(str)) for name, frame in parts.items()}
    assert session_sets["train"].isdisjoint(session_sets["validation"])
    assert session_sets["train"].isdisjoint(session_sets["test"])
    assert session_sets["validation"].isdisjoint(session_sets["test"])

split_summary = pd.DataFrame([
    {"dataset": dataset, "split": split, "rows": len(frame), "sessions": frame["session_id"].nunique()}
    for dataset, parts in [("reactive", reactive_parts), ("predictive", predictive_parts)]
    for split, frame in parts.items()
])
display(split_summary)

,dataset,split,rows,sessions
0,reactive,train,16012,16012
1,reactive,validation,5337,5337
2,reactive,test,5337,5337
3,predictive,train,2292,9
4,predictive,validation,1063,4
5,predictive,test,981,4


<div id="fase-5-entrenamiento" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Fase 5: Seleccion, entrenamiento y ajuste de hiperparametros</h2> </div>

Se ajusta `C` de una regresión logística balanceada. El reactivo usa GroupKFold y el predictivo StratifiedGroupKFold; el escalado vive dentro del pipeline. El threshold predictivo se elige en validación favoreciendo recall de reducción entre valores cercanos al mejor Macro F1.

In [5]:
def logistic_pipeline():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE)),
    ])

# Reactivo: búsqueda agrupada dentro de train.
reactive_search = GridSearchCV(
    logistic_pipeline(),
    {"model__C": [0.1, 1.0, 10.0]},
    scoring="f1_macro",
    cv=GroupKFold(n_splits=3),
    n_jobs=1,
    refit=True,
    error_score="raise",
)
reactive_search.fit(
    reactive_parts["train"][reactive_features],
    reactive_parts["train"]["target"],
    groups=reactive_parts["train"]["session_id"],
)

# Predictivo: búsqueda estratificada y agrupada dentro de train.
predictive_search = GridSearchCV(
    logistic_pipeline(),
    {"model__C": [0.1, 1.0, 10.0]},
    scoring="f1_macro",
    cv=StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=1,
    refit=True,
    error_score="raise",
)
predictive_search.fit(
    predictive_parts["train"][predictive_features],
    predictive_parts["train"]["target_code"],
    groups=predictive_parts["train"]["session_id"],
)

validation_probability = positive_probability(
    predictive_search.best_estimator_, predictive_parts["validation"][predictive_features]
)
predictive_threshold, threshold_table = choose_threshold(
    predictive_parts["validation"]["target_code"].to_numpy(),
    validation_probability,
    [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65],
)
display(pd.DataFrame([
    {"role": "reactive", "best_parameters": reactive_search.best_params_, "grouped_cv_macro_f1": reactive_search.best_score_, "threshold": np.nan},
    {"role": "predictive", "best_parameters": predictive_search.best_params_, "grouped_cv_macro_f1": predictive_search.best_score_, "threshold": predictive_threshold},
]))

,role,best_parameters,grouped_cv_macro_f1,threshold
0,reactive,{'model__C': 10.0},0.8194,NaN
1,predictive,{'model__C': 0.1},1.0000,0.5000


<div id="fase-6-evaluacion" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Fase 6: Evaluacion con metricas y modelos base</h2> </div>

Tras fijar configuración y threshold, cada candidato se reajusta con train y validación. DummyClassifier representa la política de clase mayoritaria. Se informa Macro F1, balanced accuracy y resultados por clase; accuracy aislada no es suficiente por el desbalance.

In [6]:
reactive_train_validation = pd.concat([reactive_parts["train"], reactive_parts["validation"]], ignore_index=True)
predictive_train_validation = pd.concat([predictive_parts["train"], predictive_parts["validation"]], ignore_index=True)

reactive_candidate = logistic_pipeline().set_params(**reactive_search.best_params_).fit(
    reactive_train_validation[reactive_features], reactive_train_validation["target"]
)
predictive_candidate = logistic_pipeline().set_params(**predictive_search.best_params_).fit(
    predictive_train_validation[predictive_features], predictive_train_validation["target_code"]
)

reactive_prediction = reactive_candidate.predict(reactive_parts["test"][reactive_features])
reactive_result = reactive_metrics(
    reactive_parts["test"]["target"].to_numpy(), reactive_prediction, ["low", "medium", "high"]
)
predictive_probability = positive_probability(predictive_candidate, predictive_parts["test"][predictive_features])
predictive_result = predictive_metrics(
    predictive_parts["test"]["target_code"].to_numpy(), predictive_probability, predictive_threshold
)

reactive_dummy = DummyClassifier(strategy="most_frequent").fit(
    reactive_train_validation[reactive_features], reactive_train_validation["target"]
)
reactive_dummy_result = reactive_metrics(
    reactive_parts["test"]["target"].to_numpy(),
    reactive_dummy.predict(reactive_parts["test"][reactive_features]),
    ["low", "medium", "high"],
)
predictive_dummy = DummyClassifier(strategy="most_frequent").fit(
    predictive_train_validation[predictive_features], predictive_train_validation["target_code"]
)
predictive_dummy_result = predictive_metrics(
    predictive_parts["test"]["target_code"].to_numpy(),
    positive_probability(predictive_dummy, predictive_parts["test"][predictive_features]),
    0.5,
)

evaluation = pd.DataFrame([
    {"role": "reactive", "model": "LogisticRegression", "macro_f1": reactive_result["macro_f1"], "balanced_accuracy": reactive_result["balanced_accuracy"]},
    {"role": "reactive", "model": "DummyClassifier", "macro_f1": reactive_dummy_result["macro_f1"], "balanced_accuracy": reactive_dummy_result["balanced_accuracy"]},
    {"role": "predictive", "model": "LogisticRegression", "macro_f1": predictive_result["macro_f1"], "balanced_accuracy": predictive_result["balanced_accuracy"]},
    {"role": "predictive", "model": "DummyClassifier", "macro_f1": predictive_dummy_result["macro_f1"], "balanced_accuracy": predictive_dummy_result["balanced_accuracy"]},
])
display(evaluation)
display(pd.DataFrame({"reactive_confusion_matrix": [reactive_result["confusion_matrix"]], "predictive_confusion_matrix": [predictive_result["confusion_matrix"]]}))
assert evaluation.groupby("role").apply(lambda group: group.iloc[0]["macro_f1"] > group.iloc[1]["macro_f1"], include_groups=False).all()

,role,model,macro_f1,balanced_accuracy
0,reactive,LogisticRegression,0.8318,0.9303
1,reactive,DummyClassifier,0.3017,0.3333
2,predictive,LogisticRegression,1.0000,1.0000
3,predictive,DummyClassifier,0.4891,0.5000


,reactive_confusion_matrix,predictive_confusion_matrix
0,"[[522, 13, 1], [21, 360, 9], [0, 468, 3943]]","[[42, 0], [0, 939]]"


<div id="fase-7-guardado" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Fase 7: Guardado y carga del modelo</h2> </div>

Los candidatos se guardan en un directorio temporal fuera del repositorio. Se recargan y se exige que conserven exactamente las predicciones; así se prueba serialización sin alterar el registro oficial ni dejar artefactos de ejecución.

In [7]:
TEMP_ARTIFACT_DIR = Path(tempfile.mkdtemp(prefix="streamml-agent-notebook-"))
reactive_candidate_path = TEMP_ARTIFACT_DIR / "reactive_candidate.joblib"
predictive_candidate_path = TEMP_ARTIFACT_DIR / "predictive_candidate.joblib"
joblib.dump(reactive_candidate, reactive_candidate_path)
joblib.dump(predictive_candidate, predictive_candidate_path)

reactive_loaded = joblib.load(reactive_candidate_path)
predictive_loaded = joblib.load(predictive_candidate_path)
assert np.array_equal(
    reactive_loaded.predict(reactive_parts["test"][reactive_features]), reactive_prediction
)
assert np.allclose(
    positive_probability(predictive_loaded, predictive_parts["test"][predictive_features]),
    predictive_probability,
)
print("Guardado y carga verificados en:", TEMP_ARTIFACT_DIR)

Guardado y carga verificados en: C:\Users\kenny\AppData\Local\Temp\streamml-agent-notebook-ggjd16q5


<div id="fase-8-agente" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Fase 8: Integracion del modelo con el agente</h2> </div>

La política recibe salidas ya validadas. Un riesgo predictivo tiene prioridad para reducir un nivel; los aumentos exigen confirmaciones; una capacidad insuficiente limita la recomendación; y la pérdida total activa respaldo tras un margen configurable.

In [8]:
real_reactive_row = reactive_parts["test"].iloc[0]
real_predictive_row = predictive_parts["test"].iloc[0]
reactive_recommendation = str(reactive_loaded.predict(pd.DataFrame([real_reactive_row[reactive_features]]))[0])
real_predictive_probability = float(
    positive_probability(predictive_loaded, pd.DataFrame([real_predictive_row[predictive_features]]))[0]
)
predictive_decision = "downgrade_needed" if real_predictive_probability >= predictive_threshold else "maintain"

agent = AutonomousStreamingAgent()
agent_state = AgentState(current_profile="high")
agent_decision = agent.decide(
    agent_state,
    AgentInput(
        observed_at=10_000.0,
        signal_available=True,
        reactive_profile=reactive_recommendation,
        predictive_decision=predictive_decision,
        downgrade_probability=real_predictive_probability,
        capacity_mbps=float(real_predictive_row["throughput_mean"]),
    ),
)
display(pd.DataFrame([agent_decision.to_dict()]))
assert agent_decision.reason_code

,action,current_profile,target_profile,backup_active,reason,reason_code,operational_state,apply_profile,apply_backup,target_profile_spec
0,maintain,high,high,False,Los modelos permiten mantener el perfil actual.,models_support_current_profile,stable,False,False,"{'level': 3, 'width': 1920, 'height': 1080, 'f..."


<div id="fase-9-ejemplos" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Fase 9: Ejemplos reales de uso</h2> </div>

Las filas siguientes provienen de sesiones de test. Se muestran entrada, etiqueta real, predicción y probabilidad; no son escenarios inventados.

In [9]:
reactive_real_examples = reactive_parts["test"].groupby("target", group_keys=False).head(1).copy()
reactive_real_examples["prediction"] = reactive_loaded.predict(reactive_real_examples[reactive_features])

predictive_real_examples = predictive_parts["test"].groupby("target", group_keys=False).head(1).copy()
example_probability = positive_probability(predictive_loaded, predictive_real_examples[predictive_features])
predictive_real_examples["prob_downgrade_needed"] = example_probability
predictive_real_examples["prediction"] = np.where(example_probability >= predictive_threshold, "downgrade_needed", "maintain")

display(reactive_real_examples[["session_id", *reactive_features, "target", "prediction"]])
display(predictive_real_examples[["session_id", "throughput_mean", "current_profile", "target", "prob_downgrade_needed", "prediction"]])

,session_id,upload_mbps,download_mbps,latency_ms,target,prediction
21349,O1d7ee6cb-28c6-4c5a-b3a6-7df8707dc9fb,22.8180,24.0640,26.0000,high,high
21350,O6b9bbdbb-5374-4319-96fd-117b2cc9db4b,2.2890,47.5930,30.0000,low,low
21363,O9c8634c4-3347-4644-b44a-13903c4bbfd7,4.6650,0.2880,98.0000,medium,medium


,session_id,throughput_mean,current_profile,target,prob_downgrade_needed,prediction
0,run_4191,25.0000,2,maintain,0.0007,maintain
545,run_4686,1.2138,1,downgrade_needed,0.9995,downgrade_needed


<div id="fase-10-errores" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Fase 10: Manejo de errores y casos limite</h2> </div>

Se prueban entradas inválidas, pérdida total y recuperación. La política falla de forma explícita ante probabilidades imposibles y no restaura el vivo hasta confirmar estabilidad.

In [10]:
edge_agent = AutonomousStreamingAgent(AgentPolicy(signal_loss_grace_seconds=0, recovery_stable_seconds=2))
edge_state = AgentState(current_profile="low")
edge_decisions = [
    edge_agent.decide(edge_state, AgentInput(observed_at=20_000.0, signal_available=False)),
    edge_agent.decide(edge_state, AgentInput(observed_at=20_001.0, signal_available=True)),
    edge_agent.decide(edge_state, AgentInput(observed_at=20_003.0, signal_available=True)),
]
display(pd.DataFrame([decision.to_dict() for decision in edge_decisions])[
    ["action", "backup_active", "reason_code", "reason"]
])
assert [decision.action for decision in edge_decisions] == ["switch_to_backup", "maintain_backup", "restore_live"]

try:
    AgentInput(observed_at=20_004.0, signal_available=True, capacity_mbps=-1)
except ValueError as error:
    print("Caso límite rechazado correctamente:", error)
else:
    raise AssertionError("Una capacidad negativa debía producir ValueError.")

shutil.rmtree(TEMP_ARTIFACT_DIR)
assert not TEMP_ARTIFACT_DIR.exists()
print("Artefactos temporales eliminados.")

,action,backup_active,reason_code,reason
0,switch_to_backup,True,signal_loss_confirmed,Pérdida total confirmada; activación automátic...
1,maintain_backup,True,recovery_hysteresis,Señal recuperada; esperando estabilidad antes ...
2,restore_live,False,live_signal_stable,Señal principal estable; restauración automáti...


Caso límite rechazado correctamente: capacity_mbps must be finite and non-negative.
Artefactos temporales eliminados.


<div id="conclusiones" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Conclusiones y proximos pasos</h2> </div>

El notebook valida el ciclo completo: datos y contratos, preprocesamiento sin fuga, división agrupada, ajuste de hiperparámetros, evaluación contra baselines, serialización, recarga e integración con el agente. Los ejemplos de inferencia proceden de test y los casos límite prueban respaldo, recuperación y validación estricta.

La principal limitación es experimental: el predictivo contiene solo 17 sesiones, fuerte desbalance, ventanas solapadas y muchos vectores repetidos. Antes de afirmar rendimiento de producción se deben recopilar sesiones IRL independientes, añadir jitter y pérdida a un nuevo contrato, medir QoE real y comparar estadísticamente agente completo, control reactivo y perfil fijo. La arquitectura ya permite sustituir modelos sin cambiar la política operacional.

<div id="referencias" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>Referencias y enlaces</h2> </div>

1. Contratos, manifiestos, métricas y auditorías incluidos en este repositorio StreamML.
2. RTR-NetzTest Open Data, fuente del dataset reactivo.
3. YouTube Dataset on Mobile Streaming for Internet Traffic Modeling, fuente del dataset predictivo.
